In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from compton_data_repair import data_repair
MY_JURISDICTION = "Compton"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_la_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 0.0% -> 0.0%
PERMIT_DATE available (all): 79.1% -> 79.1%
FINAL_DATE available (all): 0.0% -> 0.0%
PERMIT_DATE available (active/final): 94.4% -> 93.3%
FINAL_DATE available (final): 0.0% -> 0.0%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Active       1133
Final         384
In Review     291
Inactive       78
Name: count, dtype: int64
STATUS_NORMALIZED
Active       1189
Final         389
In Review     293
Inactive      128
Name: count, dtype: int64


In [5]:
mask = sub_df_filled["FINAL_DATE"].isna() & (sub_df_filled["STATUS_NORMALIZED"] == "Final")
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Final    *Filled: nan*
RECORD_TYPE_ORIGINAL: Mechanical Permit
FILE_DATE: None       *Filled: nan*
PERMIT_DATE: 2022-09-01   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "Status": "Finaled",
  "Issue Date": "09/01/2022"
}


In [6]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "Status": "Finaled",
  "Address ": "119 S ROSE AVE",
  "Permit #": "M22-000419",
  "Sub Type": "Alterations",
  "Issue Date": "09/01/2022",
  "Permit Type": "Mechanical Permit",
  "Work Description": "Change out single wall furnace "
}
